In [ ]:
import pandas as pd 
network_summary_stats = pd.read_csv('/home/shruti/Data/exported_data/BatchNetwork_MediaDensityT2/20260109_110008/csv/network_summary_metrics.csv')

network_summary_stats.head()
network_summary_stats.columns

In [ ]:
from math import sqrt
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import os

# Load the network summary stats
network_summary_stats = pd.read_csv('/home/shruti/Data/exported_data/BatchNetwork_MediaDensityT2/20260109_110008/csv/network_summary_metrics.csv')

# Create condition and density columns
def get_condition_and_density(well_group_name):
    if pd.isna(well_group_name) or str(well_group_name) == 'nan':
        return None, None
    wg = str(well_group_name).strip()
    if 'NBP' in wg:
        condition = 'NBP'
    elif 'DMEM' in wg:
        condition = 'DMEM'
    else:
        return None, None
    import re
    match = re.search(r'(\d+K)', wg)
    if match: 
        density = match.group(1)
    else: 
        return None, None
    return condition, density

network_summary_stats[['Condition', 'Density']] = network_summary_stats['Well Group Name'].apply(
    lambda x: pd.Series(get_condition_and_density(x))
)
network_summary_stats = network_summary_stats.dropna(subset=['Condition', 'Density'])

# Identify metric columns
metadata_columns = ['Instance', 'Date', 'Wellplate ID', 'Well Number', 'Assay Run ID', 
                   'Assay Tag', 'DIV [days]', 'Well Group Name', 'Well Group Color', 
                   'Plating Date', 'Control', 'Condition', 'Density']

metric_columns = [col for col in network_summary_stats.columns 
                 if col not in metadata_columns 
                 and pd.api.types.is_numeric_dtype(network_summary_stats[col])]

print(f"Found {len(metric_columns)} metric columns to plot")

# Get unique DIVs
div = sorted(network_summary_stats['DIV [days]'].unique())

# Get unique densities for each condition and sort them
def extract_density_number(density_str):
    """Extract numeric value from density string like '60K' -> 60"""
    numeric_part = ''.join(filter(str.isdigit, density_str))
    return int(numeric_part) if numeric_part else 0

nbp_densities = sorted(
    network_summary_stats[network_summary_stats['Condition'] == 'NBP']['Density'].unique(),
    key=extract_density_number
)
dmem_densities = sorted(
    network_summary_stats[network_summary_stats['Condition'] == 'DMEM']['Density'].unique(),
    key=extract_density_number
)

# Combine all genotypes in order: NBP densities (ascending), then DMEM densities (ascending)
all_genotypes = [f"NBP {d}" for d in nbp_densities] + [f"DMEM {d}" for d in dmem_densities]

# Create colors: shades of blue for NBP, shades of orange for DMEM
nbp_base_color = np.array([31, 119, 180]) / 255
dmem_base_color = np.array([255, 127, 14]) / 255

def create_color_shades(base_color, n_shades):
    """Create n different shades of a base color"""
    shades = []
    for i in range(n_shades):
        factor = 0.6 + (0.4 * i / max(1, n_shades - 1))
        shade = base_color * factor
        shade = np.clip(shade, 0, 1)
        shades.append(shade)
    return shades

nbp_colors = create_color_shades(nbp_base_color, len(nbp_densities))
dmem_colors = create_color_shades(dmem_base_color, len(dmem_densities))
colors = nbp_colors + dmem_colors

# Plot for each metric
for metric in metric_columns:
    total_genotypes = len(all_genotypes)
    
    # Initialize output arrays for each genotype
    output_arrays = {genotype: [] for genotype in all_genotypes}
    
    # Fill data from data frame
    for i in div:
        for genotype in all_genotypes:
            condition, density = genotype.split(' ', 1)
            temp_df = network_summary_stats.loc[(network_summary_stats['DIV [days]'] == i) & 
                                                (network_summary_stats['Condition'] == condition) & 
                                                (network_summary_stats['Density'] == density)]
            output_arrays[genotype].append(np.array(temp_df[metric]))
    
    # Check if there's any data for this metric
    has_data = any(len([v for arr in arrays for v in arr if np.isfinite(v)]) > 0 
                   for arrays in output_arrays.values())
    
    if not has_data:
        print(f"Skipping {metric} - no valid data")
        continue
    
    # Dynamic bar width
    bar_width = 0.8 / total_genotypes
    gap_between_bars = 0.01
    
    # Create x-coordinates
    x_genotype = {genotype: [] for genotype in all_genotypes}
    base_x_coordinate = np.arange(len(div))
    offset = (total_genotypes * bar_width + (total_genotypes - 1) * gap_between_bars) / 2
    centered_x = base_x_coordinate - offset + bar_width / 2
    
    for i, genotype in enumerate(all_genotypes):
        x_genotype[genotype] = centered_x + i * (bar_width + gap_between_bars)
    
    # Create figure
    fig_width = max(12, len(div) * 1.5)
    fig_height = 6
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    
    # Store statistics
    mean_data_all = {}
    yerr_data_all = {}
    n_data_all = {}
    
    for i, genotype in enumerate(all_genotypes):
        y_data = output_arrays[genotype]
        
        # Calculate statistics
        mean_data = []
        yerr_data = []
        n_data = []
        
        for yi in y_data:
            valid_values = [n for n in yi if np.isfinite(n)]
            if len(valid_values) > 0:
                mean_data.append(np.mean(valid_values))
                if len(valid_values) > 1:
                    yerr_data.append(np.std(valid_values, ddof=1) / np.sqrt(len(valid_values)))
                else:
                    yerr_data.append(0)
                n_data.append(len(valid_values))
            else:
                mean_data.append(0)
                yerr_data.append(0)
                n_data.append(0)
        
        mean_data_all[genotype] = mean_data
        yerr_data_all[genotype] = yerr_data
        n_data_all[genotype] = n_data
        
        # Plot bars
        ax.bar(x_genotype[genotype], mean_data, yerr=yerr_data, capsize=3, 
               width=bar_width, color=colors[i], edgecolor='black', linewidth=0.8,
               ecolor='black', label=genotype, alpha=0.85, error_kw={'linewidth': 1})
        
        # Plot scatter points
        jitter_amount = bar_width * 0.3
        for j in range(len(x_genotype[genotype])):
            for k in range(len(y_data[j])):
                if np.isfinite(y_data[j][k]):
                    ax.scatter(
                        x_genotype[genotype][j] + np.random.uniform(-jitter_amount, jitter_amount),
                        y_data[j][k],
                        s=20,
                        color='white',
                        marker='o',
                        edgecolors='black',
                        linewidths=0.8,
                        alpha=0.7,
                        zorder=3
                    )
    
    # Calculate significance and y-axis limits
    all_valid_values = []
    for genotype_arrays in output_arrays.values():
        for array in genotype_arrays:
            valid_vals = [v for v in array if np.isfinite(v)]
            all_valid_values.extend(valid_vals)
    
    # Track the lowest y-position for significance bars (below x-axis)
    actual_min_y_pos = 0
    
    if len(all_valid_values) > 0:
        data_max = max(all_valid_values)
        data_min = min(all_valid_values)
        data_range = data_max - data_min if data_max > data_min else data_max
        
        # Find the highest point including error bars
        highest_point = data_max
        for genotype in all_genotypes:
            for i, (mean, sem) in enumerate(zip(mean_data_all[genotype], yerr_data_all[genotype])):
                error_bar_top = mean + sem
                if error_bar_top > highest_point:
                    highest_point = error_bar_top
        
        # Perform t-tests only between matching densities - place bars BELOW x-axis
        for i in range(len(base_x_coordinate)):
            count = 1
            
            # Compare only matching densities
            all_densities = set(nbp_densities).union(set(dmem_densities))
            
            for density in all_densities:
                genotype1 = f"NBP {density}"
                genotype2 = f"DMEM {density}"
                
                if genotype1 not in all_genotypes or genotype2 not in all_genotypes:
                    continue
                
                mean1, sem1, n1 = mean_data_all[genotype1][i], yerr_data_all[genotype1][i], n_data_all[genotype1][i]
                mean2, sem2, n2 = mean_data_all[genotype2][i], yerr_data_all[genotype2][i], n_data_all[genotype2][i]
                
                if n1 < 2 or n2 < 2 or sem1 == 0 or sem2 == 0:
                    continue
                
                sed = sqrt(sem1**2.0 + sem2**2.0)
                t_stat = (mean1 - mean2) / sed
                degreef = n1 + n2 - 2
                p_value = (1.0 - stats.t.cdf(abs(t_stat), degreef)) * 2.0
                
                x1, x2 = x_genotype[genotype1][i], x_genotype[genotype2][i]
                sign = "***" if p_value <= 0.001 else "**" if p_value <= 0.01 else "*" if p_value <= 0.05 else "ns"
                
                if sign != 'ns':
                    # Place significance bars BELOW x-axis (negative y-position)
                    y_pos = -0.08 * data_range * count
                    ax.plot([x1, x2], [y_pos, y_pos], 'k', linewidth=1.2)
                    ax.text((x1 + x2) / 2, y_pos, sign, ha='center', va='top', 
                           fontsize=8, fontweight='bold')
                    
                    # Track the lowest y-position
                    actual_min_y_pos = min(actual_min_y_pos, y_pos)
                    count += 1
        
        # Set y-axis limits: top at highest error bar, bottom extends for significance bars
        y_max = highest_point + 0.05 * data_range  # Small padding above highest point
        
        if actual_min_y_pos < 0:
            # Add small padding below the lowest significance bar
            y_min = actual_min_y_pos - 0.05 * data_range
        else:
            # No significance bars, just start near zero
            y_min = max(0, data_min - 0.05 * data_range)
        
        ax.set_ylim(y_min, y_max)
    
    # Styling
    ax.set_title(f"{metric}", fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('DIV [days]', fontsize=14, fontweight='bold')
    ax.set_ylabel(f"{metric}", fontsize=14, fontweight='bold')
    ax.set_xticks(base_x_coordinate)
    ax.set_xticklabels(div, fontsize=11)
    ax.tick_params(axis='both', labelsize=10)
    
    ax.legend(title='Condition & Density', loc='upper left', bbox_to_anchor=(1.02, 1), 
              fontsize=9, title_fontsize=10, frameon=True, fancybox=True, shadow=True)
    
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    
    # Add a horizontal line at y=0 to emphasize the x-axis
    ax.axhline(y=0, color='black', linewidth=1.5, zorder=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    outdir = '/mnt/benshalom-nas/analysis/BatchTest/BatchTest_MediaDensityComparison'
    save_dir = os.path.join(outdir, 'MediaDensity_BatchTestFigures')
    os.makedirs(save_dir, exist_ok=True)

    sanitized = ''.join(ch if ch.isalnum() else '_' for ch in str(metric)).strip('_')
    save_path = os.path.join(save_dir, f"{sanitized}.png")

    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    
    print(f"Saved figure: {save_path}")
    
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
from math import sqrt
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import os

# Load the network summary stats
network_summary_stats = pd.read_csv('/home/shruti/Data/exported_data/BatchNetwork_MediaDensityT2/20260109_110008/csv/network_summary_metrics.csv')

# Create condition and density columns
def get_condition_and_density(well_group_name):
    if pd.isna(well_group_name) or str(well_group_name) == 'nan':
        return None, None
    wg = str(well_group_name).strip()
    if 'NBP' in wg:
        condition = 'NBP'
    elif 'DMEM' in wg:
        condition = 'DMEM'
    else:
        return None, None
    import re
    match = re.search(r'(\d+K)', wg)
    if match: 
        density = match.group(1)
    else: 
        return None, None
    return condition, density

network_summary_stats[['Condition', 'Density']] = network_summary_stats['Well Group Name'].apply(
    lambda x: pd.Series(get_condition_and_density(x))
)
network_summary_stats = network_summary_stats.dropna(subset=['Condition', 'Density'])

# Define metric groups - each group will create ONE comprehensive plot
metric_groups = {
    'Spikes per Burst': {
        'mean': 'Mean Spikes per Burst',
        'median': 'Median Spikes per Burst',
        'std': 'Spikes per Burst std',
        'cv': 'Spikes per Burst CV',
        'p10': 'Spikes per Burst 10th percentile',
        'p90': 'Spikes per Burst 90th percentile'
    },
    'Burst Duration': {
        'mean': 'Mean Burst Duration [s]',
        'median': 'Median Burst Duration [s]',
        'std': 'Burst Duration std [s]',
        'cv': 'Burst Duration CV',
        'p10': 'Burst Duration 10th percentile [s]',
        'p90': 'Burst Duration 90th percentile [s]'
    },
    'Burst Peak Firing Rate': {
        'mean': 'Mean Burst Peak Firing Rate [Hz]',
        'median': 'Median Burst Peak Firing Rate [Hz]',
        'std': 'Burst Peak Firing Rate std [Hz]',
        'cv': 'Burst Peak Firing Rate CV',
        'p10': 'Burst Peak Firing Rate 10th percentile [Hz]',
        'p90': 'Burst Peak Firing Rate 90th percentile [Hz]'
    },
    'IBI': {
        'mean': 'Mean IBI [s]',
        'median': 'Median IBI [s]',
        'std': 'IBI std [s]',
        'cv': 'IBI CV',
        'p10': 'IBI 10th percentile [s]',
        'p90': 'IBI 90th percentile [s]'
    },
    'ISI within Burst': {
        'mean': 'Mean ISI within Burst [ms]',
        'median': 'Median ISI within Burst [ms]',
        'std': 'ISI within Burst std [ms]',
        'cv': 'ISI within Burst CV',
        'p10': 'ISI within Burst 10th percentile [ms]',
        'p90': 'ISI within Burst 90th percentile [ms]'
    },
    'Spikes per Superburst': {
        'mean': 'Mean Spikes per Superburst',
        'median': 'Median Spikes per Superburst',
        'std': 'Spikes per Superburst std',
        'cv': 'Spikes per Superburst CV',
        'p10': 'Spikes per Superburst 10th percentile',
        'p90': 'Spikes per Superburst 90th percentile'
    },
    'Superburst Duration': {
        'mean': 'Mean Superburst Duration [s]',
        'median': 'Median Superburst Duration [s]',
        'std': 'Superburst Duration std [s]',
        'cv': 'Superburst Duration CV',
        'p10': 'Superburst Duration 10th percentile [s]',
        'p90': 'Superburst Duration 90th percentile [s]'
    }
}

# Add simple metrics (single column metrics)
simple_metrics = [
    'Network Burstiness',
    'Burst Frequency [Hz]',
    'Spikes within Bursts [%]',
    'Superburst Frequency [Hz]',
    'Spikes within Superbursts [%]'
]

# Get unique DIVs
div = sorted(network_summary_stats['DIV [days]'].unique())

# Get unique densities and sort them
def extract_density_number(density_str):
    numeric_part = ''.join(filter(str.isdigit, density_str))
    return int(numeric_part) if numeric_part else 0

nbp_densities = sorted(
    network_summary_stats[network_summary_stats['Condition'] == 'NBP']['Density'].unique(),
    key=extract_density_number
)
dmem_densities = sorted(
    network_summary_stats[network_summary_stats['Condition'] == 'DMEM']['Density'].unique(),
    key=extract_density_number
)

all_genotypes = [f"NBP {d}" for d in nbp_densities] + [f"DMEM {d}" for d in dmem_densities]

# Create colors
nbp_base_color = np.array([31, 119, 180]) / 255
dmem_base_color = np.array([255, 127, 14]) / 255

def create_color_shades(base_color, n_shades):
    shades = []
    for i in range(n_shades):
        factor = 0.6 + (0.4 * i / max(1, n_shades - 1))
        shade = base_color * factor
        shade = np.clip(shade, 0, 1)
        shades.append(shade)
    return shades

nbp_colors = create_color_shades(nbp_base_color, len(nbp_densities))
dmem_colors = create_color_shades(dmem_base_color, len(dmem_densities))
colors = nbp_colors + dmem_colors

# Function to plot comprehensive metric
def plot_comprehensive_metric(metric_name, metric_dict, df, div, genotypes, colors):
    """Create a comprehensive plot showing mean with percentile ranges"""
    
    # Use mean as primary metric, with 10th-90th percentile range
    primary_metric = metric_dict.get('mean', metric_dict.get('median'))
    p10_metric = metric_dict.get('p10')
    p90_metric = metric_dict.get('p90')
    
    if not primary_metric:
        print(f"Skipping {metric_name} - no primary metric found")
        return
    
    total_genotypes = len(genotypes)
    
    # Initialize arrays
    output_arrays = {genotype: [] for genotype in genotypes}
    p10_arrays = {genotype: [] for genotype in genotypes} if p10_metric else None
    p90_arrays = {genotype: [] for genotype in genotypes} if p90_metric else None
    
    # Fill data
    for i in div:
        for genotype in genotypes:
            condition, density = genotype.split(' ', 1)
            temp_df = df.loc[(df['DIV [days]'] == i) & 
                            (df['Condition'] == condition) & 
                            (df['Density'] == density)]
            output_arrays[genotype].append(np.array(temp_df[primary_metric]))
            if p10_arrays:
                p10_arrays[genotype].append(np.array(temp_df[p10_metric]))
            if p90_arrays:
                p90_arrays[genotype].append(np.array(temp_df[p90_metric]))
    
    # Check for data
    has_data = any(len([v for arr in arrays for v in arr if np.isfinite(v)]) > 0 
                   for arrays in output_arrays.values())
    if not has_data:
        print(f"Skipping {metric_name} - no valid data")
        return
    
    # Setup plot
    bar_width = 0.8 / total_genotypes
    gap_between_bars = 0.01
    
    x_genotype = {}
    base_x_coordinate = np.arange(len(div))
    offset = (total_genotypes * bar_width + (total_genotypes - 1) * gap_between_bars) / 2
    centered_x = base_x_coordinate - offset + bar_width / 2
    
    for i, genotype in enumerate(genotypes):
        x_genotype[genotype] = centered_x + i * (bar_width + gap_between_bars)
    
    fig_width = max(12, len(div) * 1.5)
    fig_height = 6
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    
    # Store statistics
    mean_data_all = {}
    yerr_data_all = {}
    n_data_all = {}
    
    for i, genotype in enumerate(genotypes):
        y_data = output_arrays[genotype]
        
        mean_data = []
        yerr_data = []
        n_data = []
        
        for yi in y_data:
            valid_values = [n for n in yi if np.isfinite(n)]
            if len(valid_values) > 0:
                mean_data.append(np.mean(valid_values))
                if len(valid_values) > 1:
                    yerr_data.append(np.std(valid_values, ddof=1) / np.sqrt(len(valid_values)))
                else:
                    yerr_data.append(0)
                n_data.append(len(valid_values))
            else:
                mean_data.append(0)
                yerr_data.append(0)
                n_data.append(0)
        
        mean_data_all[genotype] = mean_data
        yerr_data_all[genotype] = yerr_data
        n_data_all[genotype] = n_data
        
        # Plot bars
        ax.bar(x_genotype[genotype], mean_data, yerr=yerr_data, capsize=3, 
               width=bar_width, color=colors[i], edgecolor='black', linewidth=0.8,
               ecolor='black', label=genotype, alpha=0.85, error_kw={'linewidth': 1})
        
        # Plot percentile ranges as shaded areas if available
        if p10_arrays and p90_arrays:
            for j in range(len(x_genotype[genotype])):
                p10_vals = [v for v in p10_arrays[genotype][j] if np.isfinite(v)]
                p90_vals = [v for v in p90_arrays[genotype][j] if np.isfinite(v)]
                
                if p10_vals and p90_vals:
                    p10_mean = np.mean(p10_vals)
                    p90_mean = np.mean(p90_vals)
                    
                    # Draw vertical range line showing 10th-90th percentile
                    ax.plot([x_genotype[genotype][j], x_genotype[genotype][j]], 
                           [p10_mean, p90_mean], 
                           color=colors[i], linewidth=2, alpha=0.3, zorder=1)
    
    # Calculate y-axis limits and significance
    all_valid_values = []
    for genotype_arrays in output_arrays.values():
        for array in genotype_arrays:
            valid_vals = [v for v in array if np.isfinite(v)]
            all_valid_values.extend(valid_vals)
    
    actual_min_y_pos = 0
    
    if len(all_valid_values) > 0:
        data_max = max(all_valid_values)
        data_min = min(all_valid_values)
        data_range = data_max - data_min if data_max > data_min else data_max
        
        # Find highest point including error bars
        highest_point = data_max
        for genotype in genotypes:
            for i, (mean, sem) in enumerate(zip(mean_data_all[genotype], yerr_data_all[genotype])):
                error_bar_top = mean + sem
                if error_bar_top > highest_point:
                    highest_point = error_bar_top
        
        # Perform t-tests between matching densities
        for i in range(len(base_x_coordinate)):
            count = 1
            all_densities = set(nbp_densities).union(set(dmem_densities))
            
            for density in all_densities:
                genotype1 = f"NBP {density}"
                genotype2 = f"DMEM {density}"
                
                if genotype1 not in genotypes or genotype2 not in genotypes:
                    continue
                
                mean1 = mean_data_all[genotype1][i]
                sem1 = yerr_data_all[genotype1][i]
                n1 = n_data_all[genotype1][i]
                mean2 = mean_data_all[genotype2][i]
                sem2 = yerr_data_all[genotype2][i]
                n2 = n_data_all[genotype2][i]
                
                if n1 < 2 or n2 < 2 or sem1 == 0 or sem2 == 0:
                    continue
                
                sed = sqrt(sem1**2.0 + sem2**2.0)
                t_stat = (mean1 - mean2) / sed
                degreef = n1 + n2 - 2
                p_value = (1.0 - stats.t.cdf(abs(t_stat), degreef)) * 2.0
                
                x1, x2 = x_genotype[genotype1][i], x_genotype[genotype2][i]
                sign = "***" if p_value <= 0.001 else "**" if p_value <= 0.01 else "*" if p_value <= 0.05 else "ns"
                
                if sign != 'ns':
                    y_pos = -0.08 * data_range * count
                    ax.plot([x1, x2], [y_pos, y_pos], 'k', linewidth=1.2)
                    ax.text((x1 + x2) / 2, y_pos, sign, ha='center', va='top', 
                           fontsize=8, fontweight='bold')
                    actual_min_y_pos = min(actual_min_y_pos, y_pos)
                    count += 1
        
        # Set y-axis limits
        y_max = highest_point + 0.05 * data_range
        if actual_min_y_pos < 0:
            y_min = actual_min_y_pos - 0.05 * data_range
        else:
            y_min = max(0, data_min - 0.05 * data_range)
        ax.set_ylim(y_min, y_max)
    
    # Styling
    ax.set_title(f"{metric_name} (with 10th-90th percentile ranges)", 
                fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('DIV [days]', fontsize=14, fontweight='bold')
    ax.set_ylabel(f"{metric_name}", fontsize=14, fontweight='bold')
    ax.set_xticks(base_x_coordinate)
    ax.set_xticklabels(div, fontsize=11)
    ax.tick_params(axis='both', labelsize=10)
    
    ax.legend(title='Condition & Density', loc='upper left', bbox_to_anchor=(1.02, 1), 
              fontsize=9, title_fontsize=10, frameon=True, fancybox=True, shadow=True)
    
    ax.grid(axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.axhline(y=0, color='black', linewidth=1.5, zorder=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    return fig

# Create output directory
outdir = '/mnt/benshalom-nas/analysis/BatchTest/BatchTest_MediaDensityComparison'
save_dir = os.path.join(outdir, 'MediaDensity_ComprehensiveFigures')
os.makedirs(save_dir, exist_ok=True)

# Plot comprehensive metrics
print("Creating comprehensive metric plots...")
for metric_name, metric_dict in metric_groups.items():
    print(f"Plotting {metric_name}...")
    fig = plot_comprehensive_metric(metric_name, metric_dict, network_summary_stats, 
                                   div, all_genotypes, colors)
    if fig:
        sanitized = ''.join(ch if ch.isalnum() else '_' for ch in metric_name).strip('_')
        save_path = os.path.join(save_dir, f"{sanitized}_comprehensive.png")
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
        plt.tight_layout()
        plt.close(fig)

# Plot simple metrics
print("\nCreating simple metric plots...")
for metric in simple_metrics:
    if metric not in network_summary_stats.columns:
        continue
    print(f"Plotting {metric}...")
    fig = plot_comprehensive_metric(metric, {'mean': metric}, network_summary_stats,
                                   div, all_genotypes, colors)
    if fig:
        sanitized = ''.join(ch if ch.isalnum() else '_' for ch in metric).strip('_')
        save_path = os.path.join(save_dir, f"{sanitized}.png")
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
        plt.tight_layout()
        plt.close(fig)

print("\nAll comprehensive plots completed!")